<font size=15 color="orange"><b> Calibración_Modelo_heston_Para_OpcionesMexicanas_IPC </b></font>

In [21]:
# Celda 1: Importación de librerías
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution, minimize, Bounds
from scipy import integrate, stats
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Celda 2: Teorema de Girsanov y cambio de medida

TEOREMA DE GIRSANOV EN EL MODELO HESTON

El teorema de Girsanov permite cambiar de la medida física ℙ a la medida neutral al riesgo ℚ.
En el modelo Heston, bajo ℙ:
$$
dS_t = μS_t dt + √v_t S_t dW^ℙ_{1,t}
dv_t = κ(θ - v_t)dt + σ√v_t dW^ℙ_{2,t}
$$
Aplicando Girsanov:
$$
dW^ℚ_{1,t} = dW^ℙ_{1,t} + λ_S dt
dW^ℚ_{2,t} = dW^ℙ_{2,t} + λ_v dt
$$
Donde $λ_S = (μ - r)/√v_t y λ_v = λ/σ √v_t$

Bajo ℚ:
$$
dS_t = rS_t dt + √v_t S_t dW^ℚ_{1,t}
dv_t = κ^ℚ(θ^ℚ - v_t)dt + σ√v_t dW^ℚ_{2,t}
$$
Con:
$$
κ^ℚ = κ + λ
θ^ℚ = κθ/(κ + λ)
ρ^ℚ = ρ
$$
Esto permite valuar opciones bajo la medida neutral al riesgo.


# Celda 3: Función característica del modelo Heston
FUNCIÓN CARACTERÍSTICA DEL MODELO HESTON

La solución del modelo Heston se obtiene mediante funciones características.
Para una opción call europea:
$
C(S₀, K, v₀, τ) = S₀P₁ - Ke^{-rτ}P₂
$
Donde P₁ y P₂ son probabilidades neutrales al riesgo calculadas como:
$$
P_j = 1/2 + 1/π ∫₀^∞ Re[ e^{-iφ ln K} φ_j(φ) / (iφ) ] dφ, j ∈ {1,2}
$$
Las funciones características tienen la forma:
$$
φ_j(φ) = exp(C(τ;φ) + D(τ;φ)v₀ + iφ ln S₀)
$$
Con:
$$
C(τ;φ) = rφiτ + (a/σ²)[(b_j - ρσφi + d)τ - 2 ln((1 - ge^{dτ})/(1 - g))]
D(τ;φ) = (b_j - ρσφi + d)/σ² * (1 - e^{dτ})/(1 - ge^{dτ})
$$
Donde:
$$
d = √((ρσφi - b_j)² - σ²(2u_jφi - φ²))
g = (b_j - ρσφi + d)/(b_j - ρσφi - d)
a = κθ
b₁ = κ + λ - ρσ, b₂ = κ + λ
u₁ = 1/2, u₂ = -1/2
$$
Esta formulación permite precios semi-analíticos mediante integración numérica.



In [22]:
# Celda 4: Implementación de la función característica
def heston_charfunc(phi, S0, v0, kappa, theta, sigma, rho, lambd, tau, r):
    """
    Implementa la función característica del modelo Heston.
    
    Parámetros:
    phi: variable de integración compleja
    S0: precio spot del subyacente
    v0: varianza inicial
    kappa: velocidad de reversión a la media
    theta: varianza de largo plazo
    sigma: volatilidad de la volatilidad
    rho: correlación entre activo y volatilidad
    lambd: premio por riesgo de volatilidad
    tau: tiempo hasta vencimiento (años)
    r: tasa libre de riesgo
    """
    a = kappa * theta
    b = kappa + lambd
    rspi = rho * sigma * phi * 1j
    
    # Cálculo de d (discriminante)
    d = np.sqrt((rho * sigma * phi * 1j - b)**2 + (phi * 1j + phi**2) * sigma**2)
    
    # Cálculo de g
    g = (b - rspi + d) / (b - rspi - d)
    
    # Componentes exponenciales
    exp1 = np.exp(r * phi * 1j * tau)
    term2 = S0**(phi * 1j) * ((1 - g * np.exp(d * tau)) / (1 - g))**(-2 * a / sigma**2)
    exp2 = np.exp(a * tau * (b - rspi + d) / sigma**2 + 
                  v0 * (b - rspi + d) * ((1 - np.exp(d * tau)) / (1 - g * np.exp(d * tau))) / sigma**2)
    
    return exp1 * term2 * exp2

In [33]:
# Celda 5: Descarga de datos reales del IPC mexicano - CORREGIDA
def descargar_datos_ipc():
    """
    Descarga datos históricos del IPC mexicano y genera datos sintéticos
    de opciones basados en el comportamiento del mercado.
    
    Nota: Yahoo Finance no provee datos de opciones para el IPC mexicano.
    Por lo tanto, generamos datos sintéticos realistas basados en:
    1. Precio spot real del IPC
    2. Volatilidad histórica del IPC
    3. Parámetros típicos del mercado mexicano
    """
    
    # Descargar datos históricos del IPC
    print("Descargando datos del IPC mexicano desde Yahoo Finance...")
    ticker = "^MXX"  # Símbolo del IPC en Yahoo Finance
    ipc = yf.Ticker(ticker)
    
    # Obtener datos de los últimos 60 días
    end_date = datetime.now()
    start_date = end_date - timedelta(days=60)
    
    try:
        hist = ipc.history(start=start_date, end=end_date)
        if len(hist) == 0:
            raise ValueError("No se pudieron obtener datos históricos")
    except Exception as e:
        print(f"Error descargando datos: {e}")
        print("Usando datos de respaldo...")
        # Datos de respaldo si falla la descarga
        hist = pd.DataFrame({
            'Close': [52000, 52500, 51500, 53000, 53500],
            'Volume': [1000000, 1200000, 900000, 1100000, 1300000]
        })
    
    # Calcular precio spot (último cierre)
    S0 = float(hist['Close'].iloc[-1])
    
    # Calcular volatilidad histórica (anualizada)
    returns = np.log(hist['Close'] / hist['Close'].shift(1)).dropna()
    vol_historica = returns.std() * np.sqrt(252)  # Anualización
    
    print(f"\nDatos del IPC mexicano:")
    print(f"Precio spot actual: {S0:,.2f}")
    print(f"Volatilidad histórica (30 días): {vol_historica:.2%}")
    print(f"Rango de precios: {hist['Close'].min():,.2f} - {hist['Close'].max():,.2f}")
    print(f"Volumen promedio: {hist['Volume'].mean():,.0f}")
    
    # Parámetros reales del modelo (desconocidos en la práctica)
    params_reales = {
        'kappa': 1.8,      # Velocidad de reversión a la media (más alta para mercados emergentes)
        'theta': vol_historica**2 + 0.02,  # Varianza de largo plazo (basada en vol histórica + prima)
        'sigma': 0.35,     # Volatilidad de la volatilidad (mayor que en mercados desarrollados)
        'rho': -0.65,      # Correlación negativa típica en equity
        'lambd': 0.15,     # Premio por riesgo de volatilidad (mayor en mercados emergentes)
        'v0': vol_historica**2 * 1.2  # Varianza inicial (20% mayor que la histórica)
    }
    
    # Generar strikes realistas (basados en el precio spot)
    moneyness_range = np.linspace(0.85, 1.15, 12)  # ±15% del spot
    strikes = S0 * moneyness_range
    
    # Madureces típicas del mercado mexicano (en días)
    maturities_days = np.array([7, 14, 30, 60, 90, 180, 270, 365])
    maturities = maturities_days / 365.0  # Convertir a años
    
    # Tasa libre de riesgo mexicana (CETES 28 días)
    r = 0.112  # 11.2% anual (tasa típica en México)
    
    print(f"\nGenerando datos de opciones sintéticos...")
    print(f"Número de strikes: {len(strikes)}")
    print(f"Número de madurezas: {len(maturities)}")
    print(f"Tasa libre de riesgo: {r:.2%}")
    
    # Generar precios de opciones
    precios = []
    strikes_list = []
    maturities_list = []
    tipos_opcion = []
    
    # Parámetros para ruido dependiente de la liquidez
    # En mercados emergentes, las opciones OTM y de corto plazo son menos líquidas
    ruido_base = 0.02  # 2% de ruido base
    
    for tau in maturities:
        for K in strikes:
            # Precio teórico
            precio_teorico = heston_price(
                S0, K, params_reales['v0'],
                params_reales['kappa'],
                params_reales['theta'],
                params_reales['sigma'],
                params_reales['rho'],
                params_reales['lambd'],
                tau, r, method='trapezoidal'
            )
            
            # Asegurar que el precio teórico sea positivo
            precio_teorico = max(precio_teorico, 1e-3)
            
            # Determinar tipo de opción (ATM, ITM, OTM)
            moneyness = K / S0
            if abs(moneyness - 1) < 0.02:
                tipo = "ATM"
                multiplicador_ruido = 1.0
            elif moneyness < 1:
                tipo = "ITM"
                multiplicador_ruido = 0.8  # Más líquidas
            else:
                tipo = "OTM"
                multiplicador_ruido = 1.5  # Menos líquidas
            
            # Ruido dependiente de liquidez
            ruido_liquidez = ruido_base * multiplicador_ruido
            
            # Mayor ruido para opciones de corto plazo (menos líquidas en México)
            if tau < 30/365:
                ruido_liquidez *= 1.3
            
            # Generar ruido asimétrico (mayor para puts en rally)
            # Asegurar que la desviación estándar sea positiva
            desviacion = max(precio_teorico * ruido_liquidez, 1e-3)
            ruido = np.random.normal(0, desviacion)
            
            # Ajuste por sesgo del mercado
            if tipo == "OTM" and K < S0:  # Puts OTM
                ruido += precio_teorico * 0.01  # Pequeño sobreprecio por protección
            
            precio_con_ruido = max(precio_teorico + ruido, 0.01 * S0)  # Mínimo 1% del spot
            
            precios.append(precio_con_ruido)
            strikes_list.append(K)
            maturities_list.append(tau)
            tipos_opcion.append(tipo)
    
    # Crear DataFrame
    datos = pd.DataFrame({
        'strike': strikes_list,
        'maturity': maturities_list,
        'price': precios,
        'rate': r,
        'type': tipos_opcion,
        'moneyness': np.array(strikes_list) / S0
    })
    
    # Añadir volumen sintético (simulando menor liquidez en OTM y corto plazo)
    datos['volume'] = np.random.lognormal(mean=5, sigma=1.5, size=len(datos))
    datos.loc[datos['type'] == 'OTM', 'volume'] *= 0.3
    datos.loc[datos['maturity'] < 30/365, 'volume'] *= 0.5
    
    print(f"\nResumen de datos generados:")
    print(f"Total de opciones: {len(datos)}")
    print(f"Distribución por tipo:")
    print(datos['type'].value_counts())
    print(f"Precio promedio: {datos['price'].mean():,.2f}")
    print(f"Rango de precios: {datos['price'].min():,.2f} - {datos['price'].max():,.2f}")
    
    return datos, S0, params_reales, hist

In [37]:
# Celda 6: Integración numérica para precio de opciones - CORREGIDA
def heston_price(S0, K, v0, kappa, theta, sigma, rho, lambd, tau, r, method='quad'):
    """
    Calcula precio de call europea usando integración numérica.
    
    Métodos disponibles:
    - 'quad': integración adaptativa de SciPy (precisa pero lenta)
    - 'trapezoidal': regla trapezoidal (rápida para muchas opciones)
    """
    
    # Validar parámetros de entrada
    if v0 <= 0 or kappa <= 0 or theta <= 0 or sigma <= 0 or tau <= 0:
        return 0.0
    
    args = (S0, v0, kappa, theta, sigma, rho, lambd, tau, r)
    
    def integrand(phi):
        # Evitar phi = 0
        if abs(phi) < 1e-10:
            return 0.0
        numerator = np.exp(r * tau) * heston_charfunc(phi - 1j, *args) - K * heston_charfunc(phi, *args)
        denominator = 1j * phi * K**(1j * phi)
        return np.real(numerator / denominator)
    
    if method == 'quad':
        # Integración adaptativa (precisa)
        try:
            integral, error = integrate.quad(integrand, 1e-10, 100, limit=1000, epsabs=1e-8)
            return max((S0 - K * np.exp(-r * tau)) / 2 + integral / np.pi, 0.0)
        except:
            return 0.0
    
    elif method == 'trapezoidal':
        # Integración trapezoidal (más rápida para calibración) - usando np.trapz
        N = 2000
        phi_grid = np.linspace(1e-10, 100, N)
        integrand_vals = []
        
        for phi in phi_grid:
            integrand_vals.append(integrand(phi))
        
        integrand_vals = np.array(integrand_vals)
        
        # Regla trapezoidal usando numpy (NO scipy.integrate.trapz)
        integral = np.trapz(integrand_vals, phi_grid)
        return max((S0 - K * np.exp(-r * tau)) / 2 + integral / np.pi, 0.0)
    
    else:
        raise ValueError(f"Método {method} no reconocido")

# Celda 7: Características de mercados emergentes
CARACTERÍSTICAS DE MERCADOS EMERGENTES (MÉXICO)

1. Menor liquidez:
   - Spreads bid-ask más amplios
   - Menor profundidad del libro de órdenes
   - Mayor impacto del tamaño de las órdenes

2. Mayor asimetría de información:
   - Participación significativa de inversionistas locales
   - Influencia de flujos internacionales (FIBRAs, ETFs)

3. Riesgo cambiario:
   - Correlación entre IPC y tipo de cambio USD/MXN
   - Impacto de políticas monetarias de la FED

4. Mayor premio por riesgo:
   - Tasas de interés más altas
   - Mayor volatilidad implícita
   - Smiles de volatilidad más pronunciados

5. Estacionalidad y efectos calendario:
   - Impacto de reportes trimestrales
   - Efecto fin de año (December effect)
   - Eventos políticos locales

Estas características afectan la calibración:
- Se requiere regularización para evitar sobreajuste
- Mayor peso a opciones más líquidas (ATM, medium-term)
- Consideración de parámetros iniciales realistas


In [26]:
# Celda 8: Función objetivo con regularización robusta
def funcion_objetivo(params, datos, S0, peso_regularizacion=0.01, metodo='trapezoidal'):
    """
    Función objetivo para calibración con regularización robusta.
    
    Incluye:
    1. Error cuadrático medio ponderado por liquidez
    2. Penalización por parámetros no económicos
    3. Regularización L2 para estabilidad
    4. Penalización por violación de restricciones de Feller
    
    Parámetros:
    params: [v0, kappa, theta, sigma, rho, lambd]
    datos: DataFrame con columnas ['strike', 'maturity', 'price', 'rate', 'volume']
    S0: precio spot del subyacente
    """
    
    v0, kappa, theta, sigma, rho, lambd = params
    
    # Restricciones económicas estrictas
    if (v0 <= 0 or v0 > 0.5 or          # Varianza entre 0% y 70% vol
        kappa <= 0 or kappa > 10 or     # Kappa positivo, no excesivo
        theta <= 0 or theta > 0.5 or    # Theta positiva, razonable
        sigma <= 0 or sigma > 1.5 or    # Sigma positiva, típicamente < 1
        rho < -0.99 or rho > 0.99 or    # Correlación válida
        lambd < -1 or lambd > 2):       # Premio por riesgo razonable
        return 1e10
    
    # Condición de Feller: 2κθ > σ² (evita varianza negativa)
    if 2 * kappa * theta <= sigma**2:
        penalizacion_feller = 100 * (sigma**2 - 2 * kappa * theta + 0.01)**2
    else:
        penalizacion_feller = 0
    
    # Calcular errores ponderados por liquidez
    errores = []
    ponderaciones = []
    
    for _, row in datos.iterrows():
        # Calcular precio del modelo
        precio_modelo = heston_price(S0, row['strike'], v0, kappa, theta,
                                    sigma, rho, lambd, row['maturity'], 
                                    row['rate'], method=metodo)
        
        # Error cuadrático
        error = (precio_modelo - row['price'])**2
        
        # Ponderación por liquidez (volumen) y moneyness
        peso_liquidez = np.log1p(row.get('volume', 1))  # Log(1 + volume)
        
        # Mayor peso a opciones ATM (más informativas)
        moneyness = row['strike'] / S0
        peso_moneyness = np.exp(-5 * (moneyness - 1)**2)  # Gaussiana centrada en ATM
        
        # Menor peso a opciones de muy corto plazo (más ruidosas)
        if row['maturity'] < 7/365:
            peso_maturity = 0.5
        elif row['maturity'] < 30/365:
            peso_maturity = 0.8
        else:
            peso_maturity = 1.0
        
        peso_total = peso_liquidez * peso_moneyness * peso_maturity
        ponderaciones.append(peso_total)
        
        errores.append(error * peso_total)
    
    # Normalizar ponderaciones
    if sum(ponderaciones) > 0:
        errores = [e / sum(ponderaciones) for e in errores]
    
    # MSE ponderado
    mse_ponderado = np.mean(errores) if errores else 1e10
    
    # Regularización L2: penalizar desviaciones de valores típicos
    penalizacion_L2 = peso_regularizacion * (
        (kappa - 2.0)**2 +           # Kappa típico ~ 2
        (theta - 0.04)**2 +          # Theta típico ~ 20% vol
        (sigma - 0.3)**2 +           # Vol-of-vol típico ~ 0.3
        (rho + 0.7)**2 +             # Correlación típica ~ -0.7
        (lambd - 0.1)**2             # Premio por riesgo típico ~ 0.1
    )
    
    # Penalización por varianza inicial muy diferente de la histórica
    if 'v0_hist' in datos.columns:
        v0_hist = datos['v0_hist'].iloc[0]
        penalizacion_v0 = 0.1 * (v0 - v0_hist)**2
    else:
        # Estimar varianza histórica implícita
        var_historica = 0.04  # Asumir 20% vol
        penalizacion_v0 = 0.1 * (v0 - var_historica)**2
    
    # Función objetivo total
    objetivo_total = (mse_ponderado + 
                     penalizacion_L2 + 
                     penalizacion_feller + 
                     penalizacion_v0)
    
    return objetivo_total

In [27]:
# Celda 9: Calibración con evolución diferencial
def calibrar_heston_differential_evolution(datos, S0, n_iteraciones=50):
    """
    Calibración robusta usando evolución diferencial con inicialización múltiple.
    
    Ventajas:
    - Búsqueda global que evita mínimos locales
    - No requiere gradientes
    - Maneja funciones no convexas
    - Robustez a ruido en los datos
    
    Parámetros:
    datos: DataFrame con precios de opciones
    S0: precio spot
    n_iteraciones: número máximo de iteraciones
    """
    
    # Límites realistas para mercado mexicano
    bounds = [
        (0.01, 0.25),      # v0: varianza inicial (10-50% vol)
        (0.5, 5.0),        # kappa: velocidad reversión
        (0.02, 0.15),      # theta: varianza largo plazo (14-39% vol)
        (0.1, 0.8),        # sigma: vol-of-vol
        (-0.9, -0.3),      # rho: correlación (negativa para equity)
        (-0.2, 0.5)        # lambd: premio por riesgo
    ]
    
    print("\nIniciando calibración con evolución diferencial...")
    print(f"Población: 15 individuos")
    print(f"Límites de parámetros:")
    nombres = ['v0', 'kappa', 'theta', 'sigma', 'rho', 'lambd']
    for i, (nombre, (min_val, max_val)) in enumerate(zip(nombres, bounds)):
        print(f"  {nombre}: [{min_val:.3f}, {max_val:.3f}]")
    
    # Función wrapper para differential_evolution
    def objetivo_de(params):
        return funcion_objetivo(params, datos, S0, peso_regularizacion=0.02)
    
    # Optimización global
    result_de = differential_evolution(
        objetivo_de,
        bounds,
        strategy='best1bin',
        maxiter=n_iteraciones,
        popsize=15,
        mutation=(0.5, 1.0),
        recombination=0.7,
        seed=42,
        disp=True,
        callback=lambda xk, convergence: print(f"Iteración {convergence}, mejor valor: {objetivo_de(xk):.6f}")
    )
    
    print(f"\nResultado evolución diferencial:")
    print(f"  Éxito: {result_de.success}")
    print(f"  Mensaje: {result_de.message}")
    print(f"  Valor función objetivo: {result_de.fun:.6f}")
    
    # Refinamiento local
    print("\nRefinando con optimización local (L-BFGS-B)...")
    result_local = minimize(
        lambda params: funcion_objetivo(params, datos, S0, peso_regularizacion=0.005),
        result_de.x,
        method='L-BFGS-B',
        bounds=bounds,
        options={
            'maxiter': 1000,
            'ftol': 1e-8,
            'gtol': 1e-6,
            'disp': False
        }
    )
    
    print(f"\nResultado refinamiento local:")
    print(f"  Éxito: {result_local.success}")
    print(f"  Valor función objetivo: {result_local.fun:.6f}")
    
    if result_local.fun < result_de.fun:
        params_opt = result_local.x
        print("  Se mejoró el resultado con refinamiento local")
    else:
        params_opt = result_de.x
        print("  Se mantiene resultado de evolución diferencial")
    
    return params_opt, result_de, result_local

In [28]:
# Celda 10: Métricas de validación
def calcular_metricas_validacion(datos, S0, params_opt):
    """
    Calcula métricas de validación para la calibración.
    
    Incluye:
    - MSE, RMSE, MAE
    - Error porcentual absoluto medio (MAPE)
    - R² (coeficiente de determinación)
    - Error por tipo de opción (ITM, ATM, OTM)
    - Error por madurez
    """
    
    v0, kappa, theta, sigma, rho, lambd = params_opt
    
    # Calcular precios del modelo
    precios_modelo = []
    for _, row in datos.iterrows():
        precio = heston_price(S0, row['strike'], v0, kappa, theta,
                             sigma, rho, lambd, row['maturity'], 
                             row['rate'], method='trapezoidal')
        precios_modelo.append(precio)
    
    precios_modelo = np.array(precios_modelo)
    precios_reales = datos['price'].values
    
    # Métricas básicas
    mse = mean_squared_error(precios_reales, precios_modelo)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(precios_reales, precios_modelo)
    
    # MAPE (evitando división por cero)
    epsilon = 1e-10
    errores_pct = np.abs((precios_reales - precios_modelo) / (precios_reales + epsilon)) * 100
    mape = np.mean(errores_pct)
    
    # R²
    ss_total = np.sum((precios_reales - np.mean(precios_reales))**2)
    ss_residual = np.sum((precios_reales - precios_modelo)**2)
    r2 = 1 - (ss_residual / ss_total) if ss_total > 0 else 0
    
    # Error por tipo de opción
    errores_por_tipo = {}
    if 'type' in datos.columns:
        for tipo in ['ITM', 'ATM', 'OTM']:
            mask = datos['type'] == tipo
            if mask.any():
                mse_tipo = mean_squared_error(precios_reales[mask], precios_modelo[mask])
                errores_por_tipo[tipo] = {
                    'MSE': mse_tipo,
                    'RMSE': np.sqrt(mse_tipo),
                    'MAE': mean_absolute_error(precios_reales[mask], precios_modelo[mask]),
                    'MAPE': np.mean(errores_pct[mask]) if mask.any() else 0,
                    'N': mask.sum()
                }
    
    # Error por madurez
    errores_por_madurez = {}
    if 'maturity' in datos.columns:
        bins = [0, 30/365, 90/365, 180/365, 1, np.inf]
        labels = ['<1M', '1-3M', '3-6M', '6-12M', '>1Y']
        
        datos['maturity_bin'] = pd.cut(datos['maturity'], bins=bins, labels=labels)
        
        for bin_label in labels:
            mask = datos['maturity_bin'] == bin_label
            if mask.any():
                errores_por_madurez[bin_label] = {
                    'MSE': mean_squared_error(precios_reales[mask], precios_modelo[mask]),
                    'N': mask.sum()
                }
    
    return {
        'precios_modelo': precios_modelo,
        'precios_reales': precios_reales,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'R2': r2,
        'errores_pct': errores_pct,
        'errores_por_tipo': errores_por_tipo,
        'errores_por_madurez': errores_por_madurez
    }

In [29]:
# Celda 11: Análisis de estabilidad con bootstrap
def analizar_estabilidad_bootstrap(datos, S0, params_opt, n_bootstrap=100):
    """
    Analiza estabilidad de parámetros mediante bootstrap.
    
    Bootstrap es crucial en mercados emergentes por:
    1. Menor cantidad de datos
    2. Mayor heterogeneidad
    3. Posibles outliers influyentes
    
    Metodología:
    1. Remuestrear datos con reemplazo
    2. Re-calibrar en cada muestra
    3. Analizar distribución de parámetros
    """
    
    print(f"\nIniciando análisis de estabilidad con {n_bootstrap} muestras bootstrap...")
    
    params_bootstrap = []
    errores_bootstrap = []
    n_success = 0
    
    bounds = [
        (0.01, 0.25),
        (0.5, 5.0),
        (0.02, 0.15),
        (0.1, 0.8),
        (-0.9, -0.3),
        (-0.2, 0.5)
    ]
    
    for i in range(n_bootstrap):
        if (i + 1) % 20 == 0:
            print(f"  Muestra {i + 1}/{n_bootstrap}")
        
        # Muestreo con reemplazo
        sample_idx = np.random.choice(len(datos), size=len(datos), replace=True)
        datos_sample = datos.iloc[sample_idx].reset_index(drop=True)
        
        try:
            # Optimización local desde el óptimo original
            result = minimize(
                lambda params: funcion_objetivo(params, datos_sample, S0, peso_regularizacion=0.01),
                params_opt,
                method='L-BFGS-B',
                bounds=bounds,
                options={'maxiter': 200, 'disp': False}
            )
            
            if result.success:
                params_bootstrap.append(result.x)
                errores_bootstrap.append(result.fun)
                n_success += 1
                
        except Exception as e:
            continue
    
    print(f"Calibraciones exitosas: {n_success}/{n_bootstrap} ({n_success/n_bootstrap*100:.1f}%)")
    
    if n_success > 10:
        params_bootstrap = np.array(params_bootstrap)
        errores_bootstrap = np.array(errores_bootstrap)
        
        # Estadísticas de parámetros
        nombres = ['v0', 'kappa', 'theta', 'sigma', 'rho', 'lambd']
        stats_params = {}
        
        for i, nombre in enumerate(nombres):
            stats_params[nombre] = {
                'mean': np.mean(params_bootstrap[:, i]),
                'std': np.std(params_bootstrap[:, i]),
                'median': np.median(params_bootstrap[:, i]),
                'q5': np.percentile(params_bootstrap[:, i], 5),
                'q95': np.percentile(params_bootstrap[:, i], 95),
                'cv': np.std(params_bootstrap[:, i]) / np.abs(np.mean(params_bootstrap[:, i]))
            }
        
        # Estadísticas de error
        stats_error = {
            'mean': np.mean(errores_bootstrap),
            'std': np.std(errores_bootstrap),
            'median': np.median(errores_bootstrap),
            'q5': np.percentile(errores_bootstrap, 5),
            'q95': np.percentile(errores_bootstrap, 95)
        }
        
        return {
            'params_bootstrap': params_bootstrap,
            'errores_bootstrap': errores_bootstrap,
            'stats_params': stats_params,
            'stats_error': stats_error,
            'success_rate': n_success / n_bootstrap
        }
    else:
        print("Error: Insuficientes calibraciones exitosas para análisis")
        return None

In [30]:
# Celda 12: Visualización de resultados
def visualizar_resultados_completos(datos, S0, params_opt, metricas, resultados_bootstrap):
    """
    Visualización completa de resultados de calibración.
    
    Genera:
    1. Gráficos de ajuste del modelo
    2. Distribución de errores
    3. Análisis de estabilidad
    4. Comparación por tipo y madurez
    """
    
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Precios modelo vs mercado
    ax1 = plt.subplot(3, 3, 1)
    scatter = ax1.scatter(datos['strike'], datos['price'], 
                         c=datos['maturity']*365, cmap='viridis',
                         alpha=0.6, s=30, label='Mercado')
    
    # Ordenar para plotting
    plot_data = datos.copy()
    plot_data['precio_modelo'] = metricas['precios_modelo']
    
    # Para cada madurez, plotear línea del modelo
    for maturity in sorted(plot_data['maturity'].unique()):
        mask = plot_data['maturity'] == maturity
        subset = plot_data[mask].sort_values('strike')
        ax1.plot(subset['strike'], subset['precio_modelo'], 
                'r-', alpha=0.7, linewidth=1)
    
    ax1.axvline(S0, color='black', linestyle='--', alpha=0.5, label=f'Spot: {S0:,.0f}')
    ax1.set_xlabel('Strike')
    ax1.set_ylabel('Precio Opción')
    ax1.set_title('Ajuste del Modelo Heston')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax1, label='Madurez (días)')
    
    # 2. Distribución de errores porcentuales
    ax2 = plt.subplot(3, 3, 2)
    n_bins = min(30, len(metricas['errores_pct']) // 10)
    ax2.hist(metricas['errores_pct'], bins=n_bins, edgecolor='black', alpha=0.7)
    ax2.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Error Relativo (%)')
    ax2.set_ylabel('Frecuencia')
    ax2.set_title(f'Distribución de Errores\nMAPE: {metricas["MAPE"]:.2f}%')
    ax2.grid(True, alpha=0.3)
    
    # 3. Error por tipo de opción
    ax3 = plt.subplot(3, 3, 3)
    if metricas['errores_por_tipo']:
        tipos = list(metricas['errores_por_tipo'].keys())
        mape_por_tipo = [metricas['errores_por_tipo'][t]['MAPE'] for t in tipos]
        n_por_tipo = [metricas['errores_por_tipo'][t]['N'] for t in tipos]
        
        bars = ax3.bar(tipos, mape_por_tipo, alpha=0.7)
        ax3.set_xlabel('Tipo de Opción')
        ax3.set_ylabel('MAPE (%)')
        ax3.set_title('Error por Tipo de Opción')
        ax3.grid(True, alpha=0.3)
        
        # Añadir etiquetas con número de opciones
        for bar, n in zip(bars, n_por_tipo):
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height,
                    f'n={n}', ha='center', va='bottom', fontsize=9)
    
    # 4. Superficie de volatilidad implícita aproximada
    ax4 = plt.subplot(3, 3, 4, projection='3d')
    
    # Calcular volatilidad implícita aproximada
    plot_data['iv_approx'] = np.sqrt(-2 * np.log(plot_data['precio_modelo'] / S0) / 
                                    plot_data['maturity'])
    
    scatter = ax4.scatter(plot_data['strike']/S0, plot_data['maturity']*365,
                         plot_data['iv_approx'], c=plot_data['iv_approx'],
                         cmap='plasma', alpha=0.7)
    
    ax4.set_xlabel('Moneyness (K/S)')
    ax4.set_ylabel('Madurez (días)')
    ax4.set_zlabel('Vol Implícita Aprox.')
    ax4.set_title('Superficie de Volatilidad del Modelo')
    
    # 5. Distribución bootstrap de parámetros (si disponible)
    if resultados_bootstrap:
        ax5 = plt.subplot(3, 3, 5)
        nombres = ['v0', 'kappa', 'theta', 'sigma', 'rho', 'lambd']
        
        # Coeficiente de variación
        cvs = [resultados_bootstrap['stats_params'][n]['cv'] * 100 for n in nombres]
        
        bars = ax5.bar(nombres, cvs, alpha=0.7)
        ax5.set_xlabel('Parámetro')
        ax5.set_ylabel('Coef. Variación (%)')
        ax5.set_title('Estabilidad de Parámetros (Bootstrap)')
        ax5.grid(True, alpha=0.3)
        
        # Añadir valores
        for bar, cv in zip(bars, cvs):
            height = bar.get_height()
            ax5.text(bar.get_x() + bar.get_width()/2., height,
                    f'{cv:.1f}%', ha='center', va='bottom', fontsize=9)
    
    # 6. Comparación de errores por madurez
    ax6 = plt.subplot(3, 3, 6)
    if metricas['errores_por_madurez']:
        madureces = list(metricas['errores_por_madurez'].keys())
        rmse_por_madurez = [np.sqrt(metricas['errores_por_madurez'][m]['MSE']) 
                           for m in madureces]
        
        bars = ax6.bar(madureces, rmse_por_madurez, alpha=0.7)
        ax6.set_xlabel('Madurez')
        ax6.set_ylabel('RMSE')
        ax6.set_title('Error por Plazo de Vencimiento')
        ax6.grid(True, alpha=0.3)
    
    # 7. QQ-Plot de errores normalizados
    ax7 = plt.subplot(3, 3, 7)
    errores = metricas['precios_reales'] - metricas['precios_modelo']
    errores_normalizados = (errores - np.mean(errores)) / np.std(errores)
    
    stats.probplot(errores_normalizados, dist="norm", plot=ax7)
    ax7.set_title('QQ-Plot de Errores Normalizados')
    ax7.grid(True, alpha=0.3)
    
    # 8. Resumen de métricas
    ax8 = plt.subplot(3, 3, 8)
    ax8.axis('off')
    
    texto_metricas = f"""
    MÉTRICAS DE CALIBRACIÓN
    
    Error Cuadrático Medio (MSE): {metricas['MSE']:.2f}
    Raíz del MSE (RMSE): {metricas['RMSE']:.2f}
    Error Absoluto Medio (MAE): {metricas['MAE']:.2f}
    Error % Absoluto Medio (MAPE): {metricas['MAPE']:.2f}%
    Coeficiente R²: {metricas['R2']:.4f}
    
    Número de opciones: {len(datos)}
    """
    
    if resultados_bootstrap:
        texto_metricas += f"""
    ESTABILIDAD (BOOTSTRAP)
    Tasa de éxito: {resultados_bootstrap['success_rate']*100:.1f}%
    Error medio bootstrap: {resultados_bootstrap['stats_error']['mean']:.4f}
    CV promedio parámetros: {np.mean([resultados_bootstrap['stats_params'][n]['cv'] 
                                     for n in ['v0', 'kappa', 'theta', 'sigma', 'rho', 'lambd']])*100:.1f}%
    """
    
    ax8.text(0.1, 0.5, texto_metricas, transform=ax8.transAxes,
            fontsize=10, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 9. Parámetros calibrados
    ax9 = plt.subplot(3, 3, 9)
    ax9.axis('off')
    
    nombres = ['v0', 'kappa', 'theta', 'sigma', 'rho', 'lambd']
    valores = params_opt
    
    texto_params = "PARÁMETROS CALIBRADOS\n\n"
    for nombre, valor in zip(nombres, valores):
        if nombre == 'v0':
            vol = np.sqrt(valor) * 100
            texto_params += f"{nombre}: {valor:.4f} (√={vol:.1f}% vol)\n"
        elif nombre == 'theta':
            vol = np.sqrt(valor) * 100
            texto_params += f"{nombre}: {valor:.4f} (√={vol:.1f}% vol)\n"
        elif nombre == 'rho':
            texto_params += f"{nombre}: {valor:.4f}\n"
        else:
            texto_params += f"{nombre}: {valor:.4f}\n"
    
    ax9.text(0.1, 0.5, texto_params, transform=ax9.transAxes,
            fontsize=10, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    plt.suptitle('Calibración Heston - IPC Mexicano: Resultados Completos', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [31]:
# Celda 13: Función principal
def ejecutar_calibracion_completa():
    """
    Función principal que ejecuta todo el pipeline de calibración.
    """
    
    print("="*80)
    print("CALIBRACIÓN ROBUSTA DEL MODELO HESTON - MERCADO MEXICANO (IPC)")
    print("="*80)
    
    # Paso 1: Descargar/generar datos
    print("\n[PASO 1] Obteniendo datos del mercado...")
    datos, S0, params_reales, hist_ipc = descargar_datos_ipc()
    
    # Paso 2: Calibración
    print("\n[PASO 2] Calibrando modelo Heston...")
    params_opt, result_de, result_local = calibrar_heston_differential_evolution(
        datos, S0, n_iteraciones=30
    )
    
    # Paso 3: Métricas de validación
    print("\n[PASO 3] Calculando métricas de validación...")
    metricas = calcular_metricas_validacion(datos, S0, params_opt)
    
    # Paso 4: Análisis de estabilidad
    print("\n[PASO 4] Realizando análisis de estabilidad (bootstrap)...")
    resultados_bootstrap = analizar_estabilidad_bootstrap(datos, S0, params_opt, n_bootstrap=50)
    
    # Paso 5: Visualización
    print("\n[PASO 5] Generando visualizaciones...")
    visualizar_resultados_completos(datos, S0, params_opt, metricas, resultados_bootstrap)
    
    # Resumen ejecutivo
    print("\n" + "="*80)
    print("RESUMEN EJECUTIVO DE CALIBRACIÓN")
    print("="*80)
    
    print(f"\n1. DATOS DEL MERCADO:")
    print(f"   • Precio spot IPC: {S0:,.2f}")
    print(f"   • Número de opciones: {len(datos)}")
    print(f"   • Rango de strikes: {datos['strike'].min():,.0f} - {datos['strike'].max():,.0f}")
    print(f"   • Madurezas: {min(datos['maturity']*365):.0f} - {max(datos['maturity']*365):.0f} días")
    
    print(f"\n2. CALIDAD DEL AJUSTE:")
    print(f"   • RMSE: {metricas['RMSE']:.2f} puntos")
    print(f"   • MAE: {metricas['MAE']:.2f} puntos")
    print(f"   • MAPE: {metricas['MAPE']:.2f}%")
    print(f"   • R²: {metricas['R2']:.4f}")
    
    print(f"\n3. PARÁMETROS CALIBRADOS:")
    nombres = ['v0', 'kappa', 'theta', 'sigma', 'rho', 'lambd']
    for nombre, valor in zip(nombres, params_opt):
        if nombre == 'v0':
            vol = np.sqrt(valor) * 100
            print(f"   • {nombre}: {valor:.4f} (vol inicial: {vol:.1f}%)")
        elif nombre == 'theta':
            vol = np.sqrt(valor) * 100
            print(f"   • {nombre}: {valor:.4f} (vol largo plazo: {vol:.1f}%)")
        elif nombre == 'rho':
            print(f"   • {nombre}: {valor:.4f}")
        else:
            print(f"   • {nombre}: {valor:.4f}")
    
    if resultados_bootstrap:
        print(f"\n4. ESTABILIDAD:")
        print(f"   • Tasa éxito bootstrap: {resultados_bootstrap['success_rate']*100:.1f}%")
        cv_promedio = np.mean([resultados_bootstrap['stats_params'][n]['cv'] 
                              for n in nombres]) * 100
        print(f"   • CV promedio parámetros: {cv_promedio:.1f}%")
    
    print(f"\n5. RECOMENDACIONES:")
    
    if metricas['MAPE'] < 5:
        print("   • ✓ Calibración EXCELENTE: MAPE < 5%")
        print("   • Aplicable para trading y cobertura")
    elif metricas['MAPE'] < 10:
        print("   • • Calibración BUENA: MAPE < 10%")
        print("   • Adecuada para valuación y análisis de riesgos")
    else:
        print("   • ✗ Calibración REGULAR: MAPE > 10%")
        print("   • Considerar recalibrar con más datos o ajustar límites")
    
    if resultados_bootstrap and cv_promedio > 20:
        print("   • ⚠ Alta variabilidad en parámetros: considerar más datos históricos")
    
    print("\n" + "="*80)
    
    return {
        'datos': datos,
        'S0': S0,
        'params_opt': params_opt,
        'metricas': metricas,
        'resultados_bootstrap': resultados_bootstrap
    }

In [38]:
# Celda 14: Ejecución del pipeline completo
if __name__ == "__main__":
    try:
        # Ejecutar calibración completa
        print("="*80)
        print("INICIANDO CALIBRACIÓN COMPLETA DEL MODELO HESTON PARA IPC MEXICANO")
        print("="*80)
        
        resultados = ejecutar_calibracion_completa()
        
        # Guardar resultados en archivo
        print("\nGuardando resultados en archivo CSV...")
        
        resultados_df = pd.DataFrame({
            'timestamp': [datetime.now()],
            'S0': [resultados['S0']],
            'v0': [resultados['params_opt'][0]],
            'kappa': [resultados['params_opt'][1]],
            'theta': [resultados['params_opt'][2]],
            'sigma': [resultados['params_opt'][3]],
            'rho': [resultados['params_opt'][4]],
            'lambd': [resultados['params_opt'][5]],
            'RMSE': [resultados['metricas']['RMSE']],
            'MAPE': [resultados['metricas']['MAPE']],
            'R2': [resultados['metricas']['R2']]
        })
        
        resultados_df.to_csv('resultados_calibracion_heston_ipc.csv', index=False)
        print("✅ Resultados guardados en 'resultados_calibracion_heston_ipc.csv'")
        
    except Exception as e:
        print(f"\n❌ Error durante la ejecución: {e}")
        print("Detalles del error:")
        import traceback
        traceback.print_exc()
        
        # Crear un archivo de resultados vacío en caso de error
        try:
            resultados_error = pd.DataFrame({
                'error': [str(e)],
                'timestamp': [datetime.now()]
            })
            resultados_error.to_csv('error_calibracion.csv', index=False)
            print("Se ha guardado un registro del error en 'error_calibracion.csv'")
        except:
            pass

INICIANDO CALIBRACIÓN COMPLETA DEL MODELO HESTON PARA IPC MEXICANO
CALIBRACIÓN ROBUSTA DEL MODELO HESTON - MERCADO MEXICANO (IPC)

[PASO 1] Obteniendo datos del mercado...
Descargando datos del IPC mexicano desde Yahoo Finance...

Datos del IPC mexicano:
Precio spot actual: 63,807.32
Volatilidad histórica (30 días): 15.70%
Rango de precios: 60,773.78 - 64,712.07
Volumen promedio: 163,127,740

Generando datos de opciones sintéticos...
Número de strikes: 12
Número de madurezas: 8
Tasa libre de riesgo: 11.20%

Resumen de datos generados:
Total de opciones: 96
Distribución por tipo:
type
ITM    40
OTM    40
ATM    16
Name: count, dtype: int64
Precio promedio: 5,863.15
Rango de precios: 638.07 - 20,920.55

[PASO 2] Calibrando modelo Heston...

Iniciando calibración con evolución diferencial...
Población: 15 individuos
Límites de parámetros:
  v0: [0.010, 0.250]
  kappa: [0.500, 5.000]
  theta: [0.020, 0.150]
  sigma: [0.100, 0.800]
  rho: [-0.900, -0.300]
  lambd: [-0.200, 0.500]
differenti

KeyboardInterrupt: 